# Graph Construction Ablation: k-NN vs Pearson (20 Runs)

**Goal.** Compare two graph construction strategies for VibrationGAT with statistical rigour: **20 independent runs** per variant, **95% bootstrap confidence intervals**, and a **two-sided Mann-Whitney U test**.

**Motivation for each strategy:**
- **k-NN (Euclidean):** connects windows with similar statistical profiles — *operating-condition similarity* topology. Windows close in statistical feature space share the same operating mode and therefore reinforce discriminative patterns through attention.
- **Pearson Top-k:** connects *channels* with the highest absolute Pearson correlation within a window — *functional inter-sensor coupling* topology. Mechanical faults generate vibration patterns that propagate through the gearbox structural paths, creating physically interpretable couplings between sensors.

**Outputs:**
- `models/ablation_knn/gat_knn_run_{01..20}.pt`
- `models/ablation_pearson/gat_pearson_run_{01..20}.pt`
- `results/comparison/graph_ablation.csv` (per-run metrics)
- `results/comparison/graph_ablation_summary.csv` (mean, std, CI 95%)
- `results/comparison/graph_ablation_meta.json`

## 1. Install PyTorch Geometric (Colab)

In [ ]:
import importlib
if not importlib.util.find_spec('torch_geometric'):
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)
    print('PyTorch Geometric installed.')
else:
    print('PyTorch Geometric already available.')

## 2. Setup and seeds

In [ ]:
import os, sys, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
from torch_geometric.loader import DataLoader as PyGDataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)

# -- Environment detection --------------------------------------------------------
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    COLAB_REPO = Path('/content/gearbox-graph-ablation-2026')
    if not COLAB_REPO.exists():
        os.system(
            'git clone https://github.com/<YOUR_USERNAME>/gearbox-graph-ablation.git '
            '/content/gearbox-graph-ablation-2026'
        )
    PROJECT_ROOT = COLAB_REPO
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.gat import VibrationGAT, EarlyStopping, make_graph_dataloaders, epoch_step_gat
from src.preprocessing import compute_statistical_features, build_knn_graph, build_pearson_graph

# -- Global seeds -----------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Project root: {PROJECT_ROOT}')

## 3. Mount Drive and locate data/processed (Colab)

**Before running this cell**, update `DRIVE_PROCESSED_PATH` to the path of the `data/processed/` folder inside your Google Drive.  
The folder must contain: `scaler.pkl`, `split_config.json`, `X_train.npy`, `X_val.npy`, `X_test.npy`, `y_train.npy`, `y_val.npy`, `y_test.npy`.

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
# Set this to the path of data/processed/ inside your Google Drive.
# Example: 'MyDrive/my-project/data/processed'
DRIVE_PROCESSED_PATH = 'MyDrive/<YOUR_PROJECT_FOLDER>/data/processed'
# ─────────────────────────────────────────────────────────────────────────────

REQUIRED_PROCESSED_FILES = {
    'scaler.pkl', 'split_config.json',
    'X_train.npy', 'X_val.npy', 'X_test.npy',
    'y_train.npy', 'y_val.npy', 'y_test.npy',
}

def is_valid_processed_dir(path):
    if not path.exists() or not path.is_dir():
        return False
    return REQUIRED_PROCESSED_FILES.issubset({p.name for p in path.iterdir() if p.is_file()})

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATA_DIR = Path('/content/drive') / DRIVE_PROCESSED_PATH
    if not is_valid_processed_dir(DATA_DIR):
        raise FileNotFoundError(
            f'Could not find a valid data/processed directory at: {DATA_DIR}\n'
            'Please update DRIVE_PROCESSED_PATH at the top of this cell.'
        )
    print(f'Drive data/processed: {DATA_DIR}')
else:
    DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
    if not is_valid_processed_dir(DATA_DIR):
        raise FileNotFoundError(f'Invalid local data directory: {DATA_DIR}')
    print(f'Local data/processed: {DATA_DIR}')

print('Files found:')
for fname in sorted(REQUIRED_PROCESSED_FILES):
    print(f'  - {fname}')

## 4. Load arrays and build both graph sets

In [ ]:
X_train = np.load(DATA_DIR / 'X_train.npy')
X_val   = np.load(DATA_DIR / 'X_val.npy')
X_test  = np.load(DATA_DIR / 'X_test.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
y_val   = np.load(DATA_DIR / 'y_val.npy')
y_test  = np.load(DATA_DIR / 'y_test.npy')

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')

# --- Baseline k-NN ---------------------------------------------------------------
K_NEIGHBOURS = 8
graphs_knn_path_train = DATA_DIR / 'graphs_train.pt'
graphs_knn_path_val   = DATA_DIR / 'graphs_val.pt'
graphs_knn_path_test  = DATA_DIR / 'graphs_test.pt'

if graphs_knn_path_train.exists() and graphs_knn_path_val.exists() and graphs_knn_path_test.exists():
    print('Found cached graphs_*.pt in data/processed. Loading (k-NN)...')
    graphs_knn_train = torch.load(graphs_knn_path_train, weights_only=False)
    graphs_knn_val   = torch.load(graphs_knn_path_val,   weights_only=False)
    graphs_knn_test  = torch.load(graphs_knn_path_test,  weights_only=False)
else:
    print('No cached graphs found. Building k-NN graphs...')
    feat_train = compute_statistical_features(X_train)
    feat_val   = compute_statistical_features(X_val)
    feat_test  = compute_statistical_features(X_test)
    graphs_knn_train = build_knn_graph(feat_train, k=K_NEIGHBOURS)
    graphs_knn_val   = build_knn_graph(feat_val,   k=K_NEIGHBOURS)
    graphs_knn_test  = build_knn_graph(feat_test,  k=K_NEIGHBOURS)

# --- Pearson Top-k variant -------------------------------------------------------
K_TOP_PEARSON = 4
print('Building Pearson graphs (Top-k inter-channel correlations)...')
graphs_p_train = build_pearson_graph(X_train, k_top=K_TOP_PEARSON)
graphs_p_val   = build_pearson_graph(X_val,   k_top=K_TOP_PEARSON)
graphs_p_test  = build_pearson_graph(X_test,  k_top=K_TOP_PEARSON)

# Attach labels
def attach_labels(graphs, y):
    for i, g in enumerate(graphs):
        g.y = torch.tensor(int(y[i]), dtype=torch.long)

for g, y in [
    (graphs_knn_train, y_train), (graphs_knn_val, y_val), (graphs_knn_test, y_test),
    (graphs_p_train,   y_train), (graphs_p_val,   y_val), (graphs_p_test,   y_test),
]:
    attach_labels(g, y)

CLASS_NAMES = ['Normal (P1)', 'Inner Race (P2)', 'Roller Element (P3)', 'Outer Race (P4)']
for split_name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f'  {split_name}: {dict(zip(range(4), np.bincount(y, minlength=4)))}')

print(f'\nSanity k-NN    | x: {graphs_knn_train[0].x.shape} | edges: {graphs_knn_train[0].edge_index.shape}')
print(f'Sanity Pearson | x: {graphs_p_train[0].x.shape}   | edges: {graphs_p_train[0].edge_index.shape}')

edge_counts_p = np.array([g.edge_index.shape[1] for g in graphs_p_train])
weights_p = torch.cat([g.edge_attr.squeeze(-1) for g in graphs_p_train[:200]]).numpy()
print(f'Pearson edges/graph: mean={edge_counts_p.mean():.1f}, min={edge_counts_p.min()}, max={edge_counts_p.max()}')
print(f'Pearson |corr| sample: mean={weights_p.mean():.3f}, p50={np.median(weights_p):.3f}, p95={np.percentile(weights_p,95):.3f}')

## 5. Hyperparameters and output directories

In [ ]:
# Architecture and training
N_CLASSES      = 4
HIDDEN         = 32
HEADS          = 2
NUM_LAYERS     = 2
DROPOUT        = 0.0
BATCH_SIZE     = 64
LR             = 1e-3
MAX_EPOCHS     = 200
PATIENCE       = 5
SCHED_PATIENCE = 3

# Multi-run
N_RUNS        = 20
BASE_RUN_SEED = 42  # seeds: 42, 43, ..., 61

# Output directories
MODELS_DIR  = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'comparison'

KNN_CKPT_DIR = MODELS_DIR / 'ablation_knn'
PEA_CKPT_DIR = MODELS_DIR / 'ablation_pearson'
for d in [KNN_CKPT_DIR, PEA_CKPT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'hidden={HIDDEN}, heads={HEADS}, layers={NUM_LAYERS}, dropout={DROPOUT}')
print(f'batch={BATCH_SIZE}, lr={LR}, max_epochs={MAX_EPOCHS}, patience={PATIENCE}')
print(f'N_RUNS={N_RUNS}, seeds={BASE_RUN_SEED}..{BASE_RUN_SEED+N_RUNS-1}')
print(f'knn checkpoints  -> {KNN_CKPT_DIR}')
print(f'pearson checkpoints -> {PEA_CKPT_DIR}')
print(f'results          -> {RESULTS_DIR}')

## 6. Training and evaluation helpers

In [ ]:
def set_all_seeds(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_variant(train_graphs, val_graphs, ckpt_path, seed):
    set_all_seeds(seed)
    n_feat = train_graphs[0].x.shape[1]
    model = VibrationGAT(
        n_feat=n_feat, n_classes=N_CLASSES,
        hidden=HIDDEN, heads=HEADS, num_layers=NUM_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)

    train_dl, val_dl = make_graph_dataloaders(
        train_graphs, val_graphs, batch_size=BATCH_SIZE, seed=seed,
    )
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=SCHED_PATIENCE
    )
    early_stop = EarlyStopping(patience=PATIENCE)

    best_val_loss = float('inf')
    best_epoch    = 0

    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss, _ = epoch_step_gat(model, train_dl, loss_fn, optimizer, train=True,  device=DEVICE)
        vl_loss, _ = epoch_step_gat(model, val_dl,   loss_fn, None,      train=False, device=DEVICE)
        scheduler.step(vl_loss)
        early_stop.step(vl_loss)
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_epoch    = epoch
            torch.save(model.state_dict(), ckpt_path)
        if early_stop.should_stop:
            break

    return {
        'n_feat':        n_feat,
        'best_epoch':    best_epoch,
        'best_val_loss': best_val_loss,
        'train_seconds': time.time() - t0,
    }


def evaluate(test_graphs, ckpt_path, n_feat):
    model = VibrationGAT(
        n_feat=n_feat, n_classes=N_CLASSES,
        hidden=HIDDEN, heads=HEADS, num_layers=NUM_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, weights_only=True, map_location=DEVICE))
    model.eval()

    loader  = PyGDataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)
    loss_fn = nn.CrossEntropyLoss()

    all_preds, all_labels, all_probs = [], [], []
    total_loss, total_n = 0.0, 0
    t0 = time.time()
    with torch.no_grad():
        for batch in loader:
            batch  = batch.to(DEVICE)
            logits = model(batch.x, batch.edge_index, batch.batch)
            total_loss += loss_fn(logits, batch.y).item() * batch.num_graphs
            total_n    += batch.num_graphs
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())
    elapsed = time.time() - t0

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.concatenate(all_probs, axis=0)

    try:
        auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    except ValueError:
        auc = float('nan')

    return {
        'test_loss':             total_loss / total_n,
        'accuracy':              accuracy_score(y_true, y_pred),
        'f1_macro':              f1_score(y_true, y_pred, average='macro'),
        'precision_macro':       precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro':          recall_score(y_true, y_pred, average='macro', zero_division=0),
        'auc_ovr_macro':         auc,
        'latency_ms_per_sample': (elapsed / total_n) * 1000.0,
    }


def bootstrap_ci(data, n_resamples=2000, confidence=0.95):
    res = stats.bootstrap(
        (np.array(data),), np.mean,
        n_resamples=n_resamples, confidence_level=confidence,
        method='percentile', random_state=42,
    )
    return float(res.confidence_interval.low), float(res.confidence_interval.high)


print('Functions defined: train_variant, evaluate, bootstrap_ci')
print(f'n_feat k-NN    : {graphs_knn_train[0].x.shape[1]}')
print(f'n_feat Pearson : {graphs_p_train[0].x.shape[1]}')

## 7. Training: 20 runs × 2 variants

In [ ]:
knn_n_feat = graphs_knn_train[0].x.shape[1]
pea_n_feat = graphs_p_train[0].x.shape[1]

training_log = []
t_total = time.time()

for run_idx in range(1, N_RUNS + 1):
    seed = BASE_RUN_SEED + run_idx - 1
    print(f'[Run {run_idx:02d}/{N_RUNS}] seed={seed}', flush=True)

    ckpt_knn = KNN_CKPT_DIR / f'gat_knn_run_{run_idx:02d}.pt'
    r_knn = train_variant(graphs_knn_train, graphs_knn_val, ckpt_knn, seed)
    print(f'  k-NN     epoch={r_knn["best_epoch"]:3d}  val_loss={r_knn["best_val_loss"]:.4f}  {r_knn["train_seconds"]:.0f}s')

    ckpt_pea = PEA_CKPT_DIR / f'gat_pearson_run_{run_idx:02d}.pt'
    r_pea = train_variant(graphs_p_train, graphs_p_val, ckpt_pea, seed)
    print(f'  Pearson  epoch={r_pea["best_epoch"]:3d}  val_loss={r_pea["best_val_loss"]:.4f}  {r_pea["train_seconds"]:.0f}s')

    training_log.append({
        'run': run_idx, 'seed': seed,
        'knn_best_epoch':    r_knn['best_epoch'],
        'knn_best_val_loss': r_knn['best_val_loss'],
        'knn_train_s':       r_knn['train_seconds'],
        'pea_best_epoch':    r_pea['best_epoch'],
        'pea_best_val_loss': r_pea['best_val_loss'],
        'pea_train_s':       r_pea['train_seconds'],
    })

print(f'\nTotal training time: {(time.time()-t_total)/60:.1f} min')
training_df = pd.DataFrame(training_log)
print(training_df[['run', 'knn_best_val_loss', 'pea_best_val_loss']].to_string(index=False))

## 8. Evaluate all 40 checkpoints on the test set

In [ ]:
per_run_metrics = []

for run_idx in range(1, N_RUNS + 1):
    seed = BASE_RUN_SEED + run_idx - 1

    m_knn = evaluate(graphs_knn_test, KNN_CKPT_DIR / f'gat_knn_run_{run_idx:02d}.pt', knn_n_feat)
    m_knn.update({'variant': 'k-NN', 'run': run_idx, 'seed': seed})

    m_pea = evaluate(graphs_p_test, PEA_CKPT_DIR / f'gat_pearson_run_{run_idx:02d}.pt', pea_n_feat)
    m_pea.update({'variant': 'Pearson', 'run': run_idx, 'seed': seed})

    per_run_metrics.extend([m_knn, m_pea])
    if run_idx % 5 == 0:
        print(f'  Evaluated runs 01-{run_idx:02d}')

runs_df = pd.DataFrame(per_run_metrics)
print('\nDescriptive statistics by variant:')
print(runs_df.groupby('variant')[['f1_macro', 'auc_ovr_macro', 'accuracy']].describe().round(4))

## 9. Statistical analysis: 95% Bootstrap CI + Mann-Whitney U

In [ ]:
f1_knn  = runs_df[runs_df['variant'] == 'k-NN']['f1_macro'].values
f1_pea  = runs_df[runs_df['variant'] == 'Pearson']['f1_macro'].values
auc_knn = runs_df[runs_df['variant'] == 'k-NN']['auc_ovr_macro'].values
auc_pea = runs_df[runs_df['variant'] == 'Pearson']['auc_ovr_macro'].values

ci_f1_knn  = bootstrap_ci(f1_knn)
ci_f1_pea  = bootstrap_ci(f1_pea)
ci_auc_knn = bootstrap_ci(auc_knn)
ci_auc_pea = bootstrap_ci(auc_pea)

# Mann-Whitney U: non-parametric, no normality assumption — appropriate for n=20
mw_f1  = stats.mannwhitneyu(f1_knn,  f1_pea,  alternative='two-sided')
mw_auc = stats.mannwhitneyu(auc_knn, auc_pea, alternative='two-sided')

print('=== 95% Bootstrap CI (n_resamples=2000, method=percentile) ===')
print(f'  k-NN    F1 : {f1_knn.mean():.4f}  CI=[{ci_f1_knn[0]:.4f}, {ci_f1_knn[1]:.4f}]  std={f1_knn.std():.5f}')
print(f'  Pearson F1 : {f1_pea.mean():.4f}  CI=[{ci_f1_pea[0]:.4f}, {ci_f1_pea[1]:.4f}]  std={f1_pea.std():.5f}')
print(f'  k-NN    AUC: {auc_knn.mean():.4f}  CI=[{ci_auc_knn[0]:.4f}, {ci_auc_knn[1]:.4f}]  std={auc_knn.std():.5f}')
print(f'  Pearson AUC: {auc_pea.mean():.4f}  CI=[{ci_auc_pea[0]:.4f}, {ci_auc_pea[1]:.4f}]  std={auc_pea.std():.5f}')
print()
print('=== Two-sided Mann-Whitney U test (alpha=0.05) ===')
for name, mw in [('F1  macro', mw_f1), ('AUC macro', mw_auc)]:
    sig = 'SIGNIFICANT' if mw.pvalue < 0.05 else 'not significant'
    print(f'  {name}: U={mw.statistic:.0f}, p={mw.pvalue:.2e}  [{sig}]')

## 10. Save results

In [ ]:
# Per-run CSV
ablation_path = RESULTS_DIR / 'graph_ablation.csv'
runs_df.to_csv(ablation_path, index=False)

# Summary CSV (mean, std, CI95)
summary_rows = []
for label, f1_vals, auc_vals in [('GAT (k-NN)', f1_knn, auc_knn), ('GAT (Pearson)', f1_pea, auc_pea)]:
    ci_f1  = bootstrap_ci(f1_vals)
    ci_auc = bootstrap_ci(auc_vals)
    summary_rows.append({
        'variant':       label,
        'f1_mean':       float(f1_vals.mean()),
        'f1_std':        float(f1_vals.std()),
        'f1_ci95_low':   ci_f1[0],
        'f1_ci95_high':  ci_f1[1],
        'auc_mean':      float(auc_vals.mean()),
        'auc_std':       float(auc_vals.std()),
        'auc_ci95_low':  ci_auc[0],
        'auc_ci95_high': ci_auc[1],
    })
summary_df = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / 'graph_ablation_summary.csv'
summary_df.to_csv(summary_path, index=False)

# Full metadata JSON
meta = {
    'n_runs': N_RUNS,
    'seed_base': BASE_RUN_SEED,
    'hyperparameters': {
        'hidden': HIDDEN, 'heads': HEADS, 'num_layers': NUM_LAYERS,
        'dropout': DROPOUT, 'batch_size': BATCH_SIZE, 'lr': LR,
        'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
    },
    'graph_builders': {
        'knn':     {'k_neighbours': K_NEIGHBOURS},
        'pearson': {'k_top': K_TOP_PEARSON, 'symmetrize': True},
    },
    'statistical_tests': {
        'method': 'Mann-Whitney U two-sided',
        'alpha': 0.05,
        'f1_macro':      {'U': float(mw_f1.statistic),  'p_value': float(mw_f1.pvalue)},
        'auc_ovr_macro': {'U': float(mw_auc.statistic), 'p_value': float(mw_auc.pvalue)},
    },
    'confidence_intervals': {
        'method': 'bootstrap percentile, n_resamples=2000, confidence=0.95, random_state=42',
        'knn':     {'f1_mean': float(f1_knn.mean()),  'f1_ci95': list(ci_f1_knn),  'auc_mean': float(auc_knn.mean()), 'auc_ci95': list(ci_auc_knn)},
        'pearson': {'f1_mean': float(f1_pea.mean()),  'f1_ci95': list(ci_f1_pea),  'auc_mean': float(auc_pea.mean()), 'auc_ci95': list(ci_auc_pea)},
    },
}
with open(RESULTS_DIR / 'graph_ablation_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f'Saved: {ablation_path}')
print(f'Saved: {summary_path}')
print(f'Saved: {RESULTS_DIR / "graph_ablation_meta.json"}')
print()
print('=== Summary for the paper ===')
for _, row in summary_df.iterrows():
    print(
        f'  {row["variant"]:14s} | '
        f'F1={row["f1_mean"]:.4f} [{row["f1_ci95_low"]:.4f},{row["f1_ci95_high"]:.4f}] | '
        f'AUC={row["auc_mean"]:.4f} [{row["auc_ci95_low"]:.4f},{row["auc_ci95_high"]:.4f}]'
    )
print(f'\n  Mann-Whitney F1:  U={mw_f1.statistic:.0f},  p={mw_f1.pvalue:.2e}')
print(f'  Mann-Whitney AUC: U={mw_auc.statistic:.0f}, p={mw_auc.pvalue:.2e}')